# Ligand library extraction + dual-Fe complex combinatorial generation

## Extract ligands from 108k tmQM complexes and recombine them into new candidate molecules

### Workflow
```
tmQM_y.csv (108k) → identify metal centers → cleave M-L bonds → extract ligand SMILES
                                            ↓
                       classify by coordinating atom (N/O/P/S/Cl...)
                                            ↓
                       select Fe-related ligands → assemble onto dual-Fe scaffolds
                                            ↓
                       output candidate SMILES → GNN screening
```


In [ ]:
# ============================================================
# Step 1: Load data
# ============================================================
import numpy as np, pandas as pd
from collections import Counter, defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdmolops
from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../chem_ml5/tmQM_y.csv', sep=';')
df = df.rename(columns={
    'HL_Gap': 'gap', 'HOMO_Energy': 'homo', 'LUMO_Energy': 'lumo',
    'Dipole_M': 'dipole', 'Metal_q': 'metal_charge',
    'Electronic_E': 'E_elec'
})
print(f'Loaded {len(df):,} complexes')
df[['CSD_code', 'SMILES']].head()


In [ ]:
# ============================================================
# Step 2: Identify metal centers + cleave M-L bonds to extract ligands
# ============================================================

# Transition-metal atomic numbers
TM_SET = set([21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,
             72,73,74,75,76,77,78,79,80])
# Common coordinating atoms
DONOR_SET = {7, 8, 15, 16, 17, 35, 53}  # N, O, P, S, Cl, Br, I


def extract_ligands(smiles):
    """Extract ligand fragments from a complex SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    # Find metal atoms
    metal_idxs = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() in TM_SET]
    if not metal_idxs: return None

    # Find all M-L bonds
    ml_bonds = []
    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        z1 = mol.GetAtomWithIdx(a1).GetAtomicNum()
        z2 = mol.GetAtomWithIdx(a2).GetAtomicNum()
        is_metal_1 = z1 in TM_SET
        is_metal_2 = z2 in TM_SET
        if is_metal_1 != is_metal_2:  # exactly one is a metal
            donor_idx = a2 if is_metal_1 else a1
            donor_z = z2 if is_metal_1 else z1
            if donor_z in DONOR_SET:
                ml_bonds.append(bond.GetIdx())

    if not ml_bonds: return None

    # Cleave M-L bonds and extract fragments
    frag_mol = Chem.FragmentOnBonds(mol, ml_bonds, addDummies=True)
    if frag_mol is None: return None

    # Get fragment SMILES
    frag_smiles_list = Chem.MolToSmiles(frag_mol).split('.')

    # Separate metal fragments and ligand fragments
    ligands = []
    metal_frag = None
    for fs in frag_smiles_list:
        f_mol = Chem.MolFromSmiles(fs)
        if f_mol is None: continue
        # Check whether it contains a metal
        has_metal = any(a.GetAtomicNum() in TM_SET for a in f_mol.GetAtoms())
        has_dummy = any(a.GetAtomicNum() == 0 for a in f_mol.GetAtoms())

        if has_metal:
            metal_frag = fs
        elif has_dummy:
            # ligand fragment: clean SMILES after removing the dummy atom
            ligands.append(fs)

    return {
        'metal_frag': metal_frag,
        'ligands': ligands,
        'n_donors': len(ml_bonds),
        'metal_types': [mol.GetAtomWithIdx(i).GetSymbol() for i in metal_idxs]
    }


# Test
test_smi = df['SMILES'].values[10]
result = extract_ligands(test_smi)
if result:
    print(f'SMILES: {test_smi[:60]}...')
    print(f'Metal: {result["metal_types"]}')
    print(f'Ligands ({len(result["ligands"])}):')
    for lig in result['ligands'][:5]:
        print(f'  {lig}')
    print(f'Total M-L bonds: {result["n_donors"]}')


In [ ]:
# ============================================================
# Step 3: Batch extraction to build the ligand library
# ============================================================
from tqdm import tqdm

# Select Fe complexes as the extraction source
fe_mask = df['SMILES'].str.contains('Fe', na=False)
df_fe = df[fe_mask].head(3000)  # extract ligands from 3000 Fe complexes
print(f'Processing {len(df_fe)} Fe complexes...')

ligand_db = defaultdict(list)  # {donor_element: [(smiles, count), ...]}
metal_scaffolds = []  # Fe scaffolds
stats = {'success': 0, 'failed': 0}

for _, row in tqdm(df_fe.iterrows(), total=len(df_fe)):
    result = extract_ligands(row['SMILES'])
    if result and result['ligands']:
        stats['success'] += 1
        if result['metal_frag']:
            metal_scaffolds.append(result['metal_frag'])
        for lig in result['ligands']:
            # identify the coordinating atom
            lmol = Chem.MolFromSmiles(lig)
            if lmol is None: continue
            for a in lmol.GetAtoms():
                if a.GetAtomicNum() == 0:  # dummy atom = coordination site
                    for nb in a.GetNeighbors():
                        donor_z = nb.GetAtomicNum()
                        if donor_z in DONOR_SET:
                            donor_sym = nb.GetSymbol()
                            # store the clean ligand with the dummy removed
                            clean_mol = Chem.DeleteSubstructs(
                                lmol, Chem.MolFromSmarts('[#0]'))
                            if clean_mol:
                                clean_smi = Chem.MolToSmiles(clean_mol)
                                ligand_db[donor_sym].append(clean_smi)
    else:
        stats['failed'] += 1

print(f'\nStats: {stats}')
print(f'Unique Fe scaffolds: {len(set(metal_scaffolds))}')
print(f'\nLigand library:')
for donor, ligs in sorted(ligand_db.items()):
    unique = set(ligs)
    print(f'  {donor}-donor: {len(ligs)} total, {len(unique)} unique')


In [ ]:
# ============================================================
# Step 4: Deduplicate + filter common ligands
# ============================================================

ligand_unique = {}
MIN_FREQ = 3  # must appear at least 3 times
for donor, ligs in ligand_db.items():
    counter = Counter(ligs)
    filtered = {smi: cnt for smi, cnt in counter.items() if cnt >= MIN_FREQ}
    ligand_unique[donor] = dict(sorted(filtered.items(), key=lambda x: -x[1])[:200])

for donor, ligs in ligand_unique.items():
    print(f'{donor}: {len(ligs)} unique ligands (freq >= {MIN_FREQ})')
    if ligs:
        top3 = list(ligs.items())[:3]
        for smi, cnt in top3:
            print(f'  [{cnt}x] {smi[:60]}')

total_ligands = sum(len(v) for v in ligand_unique.values())
print(f'\nTotal ligand pool: {total_ligands}')


In [ ]:
# ============================================================
# Step 5: Define Fe scaffolds + combinatorial generation
# ============================================================

# Define several common Fe coordination-geometry scaffolds
# [Fe] = dummy atom marking the Fe site; the number = coordination number
scaffolds = {
    'Fe_octa_6':  '[Fe]([L1])([L2])([L3])([L4])([L5])[L6]',   # 6-coordinate octahedral
    'Fe_sqplan_4': '[Fe]([L1])([L2])([L3])[L4]',               # 4-coordinate square planar
    'Fe_trig_5':   '[Fe]([L1])([L2])([L3])([L4])[L5]',         # 5-coordinate
}

# Select ligands: take top-N from each donor class
N_PER_DONOR = 10
selected_ligands = {}
for donor, ligs in ligand_unique.items():
    selected_ligands[donor] = list(ligs.keys())[:N_PER_DONOR]

for donor, ligs in selected_ligands.items():
    print(f'{donor}: {len(ligs)} ligands selected')

# Combinatorial generation (simplified: random sampling)
import random
random.seed(42)

print('\nGenerating Fe(octa_6) complexes with mixed ligands...')
generated = []
for _ in range(1000):  # generate 1000
    # randomly pick coordinating-atom types
    donor_types = random.choices(list(selected_ligands.keys()), k=6)
    ligands_chosen = []
    for dt in donor_types:
        if selected_ligands[dt]:
            ligands_chosen.append(random.choice(selected_ligands[dt]))

    if len(ligands_chosen) == 6:
        # assemble SMILES: [Fe](L1)(L2)...(L6)
        parts = ['[Fe]'] + [f'({l})' for l in ligands_chosen]
        gen_smi = ''.join(parts)
        generated.append(gen_smi)

print(f'Generated {len(generated)} candidate complexes')
print('\nExamples:')
for smi in generated[:5]:
    print(f'  {smi}')


## Next steps

The generated candidate SMILES list can be:
1. fed to the GAT GNN in tmQM_Fe.ipynb to predict the 8 DFT properties
2. rank and filter by gap/dipole/metal_charge
3. verified accurately by DFT (ORCA) for the top 10
4. submitted for experiments

---

**Current status:**
- Stage 1 (baseline models): completed ✅
- Stage 2 (ligand library + combinatorial generation): this notebook ✅
- Stage 3 (GNN screening): pending compute availability / to be replaced by the RF version
- Stage 4 (DFT validation + experiments): to be advanced
